# 🔥 guato_ba — Visão geral do estudo
### Mapeamento automático de áreas queimadas do Pantanal
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Este é o **mapa de leitura** do estudo **guato_ba**. O nome homenageia o povo **Guató**, habitantes ancestrais do Pantanal. A proposta é **registrar, ano após ano e de forma automática, onde o Pantanal queimou** — a partir de pares de imagens **Sentinel-2** de duas datas (bitemporal).

Esta coleção de notebooks documenta **tudo o que vem sendo feito**, do dado bruto ao produto final. Comece por aqui e siga a numeração.

</div>

# <span style="color:#cf4c39">🧭 1. O roteiro dos notebooks</span>
<hr style="border:1px solid #e69627;">

| # | Notebook | O que explica |
|---|---|---|
| 00 | **`00_visao_geral`** | (este) o estudo em resumo e o índice |
| 01 | **`01_datasets_A_e_B`** | os dois datasets publicados no Zenodo: como carregar e visualizar |
| 02 | **`02_modelos_deep_learning`** | as 7 arquiteturas DL (U-Net, atenção, siamês…) e a função de perda |
| 03 | **`03_modelos_machine_learning`** | os 4 modelos ML por pixel e os 11 atributos espectrais |
| 04 | **`04_metricas_e_avaliacao`** | por que **IoU da classe queimada**, e os resultados (val 2024 + OOT 2025) |
| 05 | **`05_acumulado_anual`** | como vira **produto anual** do Pantanal inteiro (tile-a-tile) |
| 06 | **`06_tutorial_ponta_a_ponta`** | tutorial reproduzível: do Brazil Data Cube à cicatriz |
| 07 | **`07_hpo_optuna_neptune`** | busca de hiperparâmetros (Optuna) e rastreio de experimentos |


# <span style="color:#cf4c39">🛰️ 2. A ideia em uma frase</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Uma **cicatriz de queimada** é uma **mudança no tempo**: a mesma área perde vigor de vegetação entre uma data **T1** (antes) e uma data **T2** (depois). O estudo aprende essa mudança a partir de **4 bandas** Sentinel-2 — **B04 (vermelho), B8A (NIR), B11 (SWIR1), B12 (SWIR2)** — em T1 e T2, e devolve uma máscara de **queimado / não-queimado**.

</div>

# <span style="color:#cf4c39">🧱 3. A regra inegociável — o guardrail do Pantanal</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

**Toda** etapa (referência, treino, validação, teste, acumulado, métricas) é **recortada ao bioma Pantanal** e **nunca** considera nada fora dele. Além disso, as entradas já vêm **sem nuvem e sem água** (filtro **SCL ∈ {4,5,7}** + índice de água **WIW**); pixels inválidos são marcados para serem **ignorados** (rótulo `255`). O modelo **nunca vê** nuvem nem água.

</div>

# <span style="color:#cf4c39">🧠 4. O que é comparado</span>
<hr style="border:1px solid #e69627;">

- **Deep learning (7)** — segmentação **bitemporal** (par T1→T2), saída por patch: `unet`, `unet_nored`, `att_unet`, `biunet`, `biatt_unet`, `resnet34_siam`, `deeplabv3plus`.
- **Machine learning (4)** — classificação **por pixel** com 11 atributos: `rf`, `xgboost`, `mlp`, `svm`.

Treina-se em **2024** e testa-se **fora da amostra (OOT)** em **2025** — o teste mais honesto: um ano que o modelo nunca viu.

# <span style="color:#cf4c39">📊 5. Os resultados em uma tabela</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

A métrica central é o **IoU da classe queimada**. A célula abaixo lê a tabela oficial do estudo (`results/artigo/modelos_ml_dl_desempenho.csv`) e mostra o teste OOT 2025, ordenado.

</div>

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT/"results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df = pd.read_csv(ROOT/"results/artigo/modelos_ml_dl_desempenho.csv")
oot = (df[df["conjunto"].str.contains("OOT")]
         .sort_values("IoU", ascending=False)
         [["modelo","tipo","IoU","F1","omissao_OE","comissao_CE"]]
         .reset_index(drop=True))
oot.round(3)

,modelo,tipo,IoU,F1,omissao_OE,comissao_CE
0,unet,DL,0.745,0.854,0.222,0.044
1,deeplabv3plus,DL,0.742,0.852,0.224,0.045
2,att_unet,DL,0.730,0.844,0.236,0.046
3,biatt_unet,DL,0.712,0.832,0.266,0.031
4,unet_nored,DL,0.706,0.827,0.265,0.041
5,biunet,DL,0.540,0.702,0.152,0.569
6,rf,ML,0.522,0.686,0.181,0.570
7,xgboost,ML,0.519,0.684,0.179,0.582
8,mlp,ML,0.485,0.653,0.185,0.681
9,svm,ML,0.477,0.646,0.173,0.732


<div style="text-align: justify; margin-right: 2.5%;">

💡 **Leitura:** as U-Nets de melhor IoU lideram no ano nunca visto; os modelos ML (por pixel) detectam bem (recall alto) mas superestimam a área (comissão maior), por não usarem contexto espacial. Os detalhes de cada número estão no notebook **04**.

</div>

# <span style="color:#cf4c39">▶️ 6. Como reproduzir</span>
<hr style="border:1px solid #e69627;">

Os dados pesados (patches, rasters, pesos) **não** ficam no git — veja **`DATA.md`**. Os **datasets A e B** estão publicados no **Zenodo** (notebook **01**). Para rodar o pipeline completo:

```bash
cd guato_ba
P=/opt/conda/envs/geospatial/bin/python
$P main.py reference      # valida o guardrail (referência ∩ bioma)
$P main.py data --forcar  # (re)constrói os patches 2024 + teste OOT 2025
$P main.py train-dl --modelo unet
$P main.py train-ml       # rf, xgboost, mlp, svm
$P main.py evaluate       # -> results/tabela_patch.csv
$P main.py accumulate-full --modelo biatt_unet --thr 0.90 --mmu 24
```

Veja o **`README.md`** para a descrição completa do protocolo.